In [1]:
!pip install -q ultralytics fastapi uvicorn python-multipart opencv-python-headless nest-asyncio

In [2]:
import torch
import cv2
import numpy as np

import threading
import time
import requests
import uvicorn
import nest_asyncio

from ultralytics import YOLO

from fastapi import FastAPI, UploadFile, File, HTTPException

from IPython.display import display, HTML
from google.colab import output

nest_asyncio.apply()

print("All libraries imported successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
All libraries imported successfully!


In [3]:
print("PyTorch Version:", torch.__version__)

if torch.cuda.is_available():
    device = 0
    print("GPU Available:", True)
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("GPU Available:", False)
    print("Running on CPU")

print("Selected Device:", device)

PyTorch Version: 2.14.1+cpu
GPU Available: False
Running on CPU
Selected Device: cpu


In [4]:
# Load pretrained YOLOv8 Nano model
model = YOLO("yolov8n.pt")

print("VisionPort YOLO model loaded successfully!")
print("Model: YOLOv8 Nano")
print("Device:", device)

VisionPort YOLO model loaded successfully!
Model: YOLOv8 Nano
Device: cpu


In [5]:
app = FastAPI(
    title="VisionPort Traffic Detection",
    description="Traffic object detection using YOLOv8 and GPU",
    version="1.0"
)

print("VisionPort FastAPI application created!")

VisionPort FastAPI application created!


In [6]:
# Home endpoint
@app.get("/")
def home():
    return {
        "project": "VisionPort",
        "status": "Running",
        "model": "YOLOv8 Nano",
        "device": str(device)
    }


# Health endpoint
@app.get("/health")
def health():
    return {
        "api": "Active",
        "gpu_available": torch.cuda.is_available()
    }


# Object detection endpoint
@app.post("/detect")
async def detect_traffic(file: UploadFile = File(...)):

    # 1. Read uploaded image
    contents = await file.read()

    if not contents:
        raise HTTPException(
            status_code=400,
            detail="Uploaded file is empty"
        )

    # 2. Convert image bytes into NumPy array
    image_array = np.frombuffer(
        contents,
        dtype=np.uint8
    )

    # 3. Decode image using OpenCV
    image = cv2.imdecode(
        image_array,
        cv2.IMREAD_COLOR
    )

    # 4. Validate image
    if image is None:
        raise HTTPException(
            status_code=400,
            detail="Invalid image"
        )

    # 5. Execute YOLO model
    results = model.predict(
        source=image,
        device=device,
        conf=0.40,
        verbose=False
    )

    # 6. Prepare result containers
    detections = []
    counts = {}

    # 7. Process detected objects
    for box in results[0].boxes:

        class_id = int(box.cls[0])
        label = model.names[class_id]
        confidence = float(box.conf[0])

        # Keep only selected traffic categories
        if label not in [
            "person",
            "bicycle",
            "car",
            "motorcycle",
            "bus",
            "truck"
        ]:
            continue

        # Extract bounding box coordinates
        coordinates = box.xyxy[0].cpu().tolist()

        # Count detected objects
        counts[label] = counts.get(label, 0) + 1

        # Store detection details
        detections.append({
            "class": label,
            "confidence": round(confidence, 3),
            "bounding_box": [
                round(value, 2) for value in coordinates
            ]
        })

    # 8. Return final JSON response
    return {
        "project": "VisionPort",
        "filename": file.filename,
        "processing_device": str(device),
        "total_objects": len(detections),
        "object_counts": counts,
        "detections": detections
    }


print("All VisionPort API endpoints created successfully!")

All VisionPort API endpoints created successfully!


In [7]:
def start_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )


server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

# Check whether the API is responding
try:
    response = requests.get("http://127.0.0.1:8000/health")

    if response.status_code == 200:
        print("VisionPort API server started successfully!")
        print("Health Status:", response.json())
    else:
        print("Server response:", response.status_code)

except requests.exceptions.RequestException as error:
    print("Server startup error:", error)

VisionPort API server started successfully!
Health Status: {'api': 'Active', 'gpu_available': False}


In [8]:
proxy_url = output.eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

display(HTML(
    f'''
    <h3>VisionPort API Documentation</h3>
    <a href="{proxy_url}/docs" target="_blank">
        Open Swagger UI
    </a>
    '''
))